<a href="https://colab.research.google.com/github/woawoal/AI-human/blob/main/22_LoRA_QLoRA_%ED%8C%8C%EC%9D%B8%ED%8A%9C%EB%8B%9D_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 22. LoRA/QLoRA 파인튜닝 — AI 휴먼 길들이기 (페르소나 챗봇 만들기)

> 선수지식: 02_HF 모델 로드·generate, 05/06 프롬프트, 16_RAG, Python 기본
>
> 오늘 한 줄: **프롬프트로 안 되던 '말투·성격'을 모델 가중치에 직접 새긴다 — 작은 모델(Qwen2.5-1.5B)에 QLoRA로 페르소나를 학습시키고, 학습 전/후를 비교한 뒤 Gradio 챗봇으로 대화한다.**

## 🎯 학습목표
- **언제 파인튜닝인가**: 지식은 RAG, 말투·성격은 파인튜닝 — 구분해서 고를 수 있다
- **LoRA**(원본 동결 + 저랭크 어댑터)와 **QLoRA**(4bit 베이스 + LoRA)의 원리를 설명할 수 있다
- **LoraConfig + TRL SFTTrainer**로 chat 데이터에 **QLoRA**를 직접 돌릴 수 있다
- **학습 전/후를 같은 질문으로 비교**하고, 학습된 페르소나를 **Gradio 챗봇**으로 띄울 수 있다

---

## 0. 환경 준비 — Colab/Kaggle 공통 + 모델 캐시(재다운로드 방지)

> 코랩은 GPU 한도/세션 종료가 잦아 **매번 대형 모델을 다시 받는** 참사가 납니다. 아래 셀이 **모델 캐시를 영구 저장소**(코랩=구글드라이브, 캐글=작업폴더)로 돌려 **1회만 다운로드**하게 합니다. **캐글에서도 그대로 실행**됩니다 — 우측 *Settings → Internet ON*, *Accelerator = GPU(T4 x2 / P100)*. 캐글은 주당 GPU 시간이 넉넉(약 30h)해 코랩 한도의 대안입니다.

In [1]:
# ── 0. 환경 자동 감지 + 모델 캐시(재다운로드 방지) — Colab / Kaggle / 로컬 공통 ──
import os, sys

def _detect_env():
    if "google.colab" in sys.modules: return "colab"
    if os.path.exists("/kaggle"):      return "kaggle"
    return "local"
ENV = _detect_env()

if ENV == "colab":
    try:
        from google.colab import drive
        drive.mount("/content/drive")                       # 한 번 인증
        CACHE = "/content/drive/MyDrive/ai_human_models"    # ★ 드라이브에 모델 캐시(영구)
    except Exception as e:
        print("드라이브 마운트 생략:", e); CACHE = "/content/hf_cache"
    WORK = "/content"
elif ENV == "kaggle":
    CACHE = "/kaggle/working/hf_cache"                       # 작업폴더(출력으로 보존)
    WORK  = "/kaggle/working"
    if not os.path.exists("/content"):                       # /content 하드코딩 호환 시도
        try: os.symlink("/kaggle/working", "/content")
        except Exception: pass
else:
    CACHE = os.path.expanduser("~/ai_human_models"); WORK = os.getcwd()

# HF 캐시를 영구 위치로 고정 → from_pretrained 가 같은 모델을 두 번 받지 않음
os.environ["HF_HOME"]      = CACHE
os.environ["HF_HUB_CACHE"] = os.path.join(CACHE, "hub")
os.makedirs(CACHE, exist_ok=True); os.makedirs(WORK, exist_ok=True)
print(f"환경={ENV} · 모델캐시={CACHE} · 작업폴더(WORK)={WORK}")
print("※ 이후 경로는 WORK 변수를 쓰세요(예: f'{WORK}/persona_adapter'). 코랩=/content, 캐글=/kaggle/working")

Mounted at /content/drive
환경=colab · 모델캐시=/content/drive/MyDrive/ai_human_models · 작업폴더(WORK)=/content
※ 이후 경로는 WORK 변수를 쓰세요(예: f'{WORK}/persona_adapter'). 코랩=/content, 캐글=/kaggle/working


---

## 🔧 환경 설정 (가장 먼저 실행) — 설치는 이 한 셀에서 모두

아래 셀을 **노트북 맨 처음에 한 번만** 실행하세요. **모든 설치를 여기서 일괄**로 합니다(중간 설치 금지). 이후 셀들은 여기서 만든 변수(`tok`, `base_model`, `MODEL_ID`)를 그대로 사용합니다.

In [2]:
# ============================================================
#  공통 환경 설정  (이 셀을 가장 먼저 실행하세요)
#  - 코랩/캐글 GPU에서 4bit 모델을 '직접' 로드 (별도 서버 불필요)
#  - 파인튜닝 풀스택: transformers + peft + trl + bitsandbytes + accelerate + datasets
# ============================================================
%pip install -q transformers peft trl bitsandbytes accelerate datasets
%pip install -q gradio   # ★ 데모/보조 패키지도 첫 셀에서 미리 (중간 설치 금지)

import os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 작은 모델로 '빠르게' 학습 한 바퀴를 돌리는 게 목적(원리는 크기와 무관하게 동일).
# OOM 폴백: 메모리가 빠듯하면 그대로 1.5B 유지 / 더 줄이려면 "Qwen/Qwen2.5-0.5B-Instruct"
MODEL_ID = os.getenv("LLM_MODEL", "Qwen/Qwen2.5-1.5B-Instruct")

# ── QLoRA의 'Q': 베이스를 4bit(nf4)로 압축해 로드 → 메모리 약 1/4 ──
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",                 # 정규분포에 맞춘 4bit (단순 반올림보다 손실↓)
    bnb_4bit_use_double_quant=True,            # 양자화 상수까지 압축(메모리 추가 절약)
    bnb_4bit_compute_dtype=torch.float16,      # 저장은 4bit, 계산은 fp16 (정밀도 절충)
)

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:                      # 일부 모델은 pad 토큰이 없음 → eos로 대체
    tok.pad_token = tok.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16,
)
print("로드 완료:", MODEL_ID, "| 디바이스:", base_model.device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.8/838.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.4 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


로드 완료: Qwen/Qwen2.5-1.5B-Instruct | 디바이스: cuda:0


> 💡 **설치는 첫 셀에서 끝.** 학습 중간에 `pip install`을 하면 런타임 충돌·재시작의 단골 원인입니다. 위 한 셀에 파인튜닝에 필요한 패키지(`transformers·peft·trl·bitsandbytes·accelerate·datasets`)와 데모용 `gradio`가 모두 들어 있습니다.

---

## 🚗 비유로 이해하기 — LoRA는 '포스트잇', QLoRA는 '압축한 책 + 포스트잇'

거대한 LLM(두꺼운 백과사전)을 통째로 다시 인쇄(풀 파인튜닝)하지 말고, 바꿀 곳에만 **얇은 포스트잇(어댑터)** 을 붙입니다.

| 개념 | 비유 | 핵심 |
|---|---|---|
| **풀 파인튜닝** | 책 통째로 재인쇄 | 가중치 100% 학습 → 큰 GPU 여러 장 |
| **LoRA** | 본문은 두고 **포스트잇**만 | 원본 동결❄ + 얇은 저랭크 어댑터(A·B)만 학습(파라미터 1% 미만) |
| **QLoRA** | **압축한 책** + 포스트잇 | 베이스를 4bit로 압축해 메모리↓ → 무료 코랩 GPU로도 학습 |
| **target_modules** | 포스트잇 붙일 위치 | 보통 어텐션 `q/k/v/o_proj` |
| **r · alpha** | 포스트잇 두께·세기 | r=어댑터 용량, alpha≈2r |

```
사용:  출력 = W·x + (B·A)·x      (W는 동결❄, 학습은 B·A만)
저랭크: ΔW(큰 변화) ≈ B(d×r)·A(r×d),  r = 8~16 (아주 작음)
QLoRA: 4bit로 압축한 W(동결) + 16bit 어댑터(학습)
```

> 🔑 **지식은 RAG, 말투·성격은 파인튜닝.** 오늘은 '말투·성격(패턴)'을 가중치에 새깁니다 — 그래서 적은 데이터(10~30개)로도 효과가 큽니다.

---

## ⌨️ 따라하기 ① — 학습 '전' 모델의 말투 확인 (베이스라인)

먼저 **학습하기 전** 모델이 어떻게 답하는지 봐 둡니다. 나중에 학습 후와 **같은 질문으로 비교**하기 위한 기준점입니다.
생성은 **HF 정석 패턴**(chat template → `tokenize` → `generate` → 새 토큰만 디코드)을 씁니다.

In [3]:
import torch

@torch.no_grad()
def chat(model, user_msg, system_msg="너는 친절한 AI 비서야.", max_new_tokens=160):
    """HF 정석 생성 패턴 — input_ids 길이만큼 잘라 '새로 생성된 토큰만' 디코드."""
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user",   "content": user_msg},
    ]
    # apply_chat_template 으로 모델이 기대하는 형식의 입력을 만든다 (딕셔너리로 받기)
    inputs = tok.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,     # ★ return_dict=True → attention_mask 포함 딕셔너리
    ).to(model.device)
    out = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=True, temperature=0.7, top_p=0.9,
        pad_token_id=tok.pad_token_id,
    )
    # ★ inputs가 딕셔너리이므로 입력 길이를 구할 때 inputs["input_ids"].shape를 사용합니다.
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(gen, skip_special_tokens=True).strip()

# 학습 전 베이스 모델의 답(밋밋한 'AI 말투'일 것)
TEST_QS = ["오늘 기분이 좀 별로야.", "주말에 뭐 하면 좋을까?", "자기소개 한 번 해줘."]
print("="*60, "\n[학습 전 / 베이스 모델]\n", "="*60)
for q in TEST_QS:
    print(f"\nQ: {q}\nA: {chat(base_model, q)}")

[학습 전 / 베이스 모델]


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Q: 오늘 기분이 좀 별로야.
A: 죄송합니다, 당신의 기분이 좋지 않다면. 어떻게 도와드릴 수 있을까요?

Q: 주말에 뭐 하면 좋을까?
A: 주말에는 다양한 활동들이 있습니다. 예를 들어, 여행, 스터디, 음악회, 게임, 영화관람, 캠핑, 토론회, 레시피 공유, 낯선 지역 탐색 등이 있습니다. 어떤 주제가 가장 재미있고 신鮮할까요? 이에 대해 좀 더 자세히 말씀해주시겠습니까?

Q: 자기소개 한 번 해줘.
A: 안녕하세요! 저는 여러분의 질문에 대답하고 도움을 드리는 역할을 하는 AI 비서입니다. 언제든지 질문이나 요청이 있으면 언제든 대화를 나누어 보세요. 어떤 도움이 필요한가요?


> 💡 이 답들을 기억해 두세요. 학습 후 **같은 질문**을 다시 던져 '말투가 바뀌었는지'를 눈으로 비교합니다(오늘의 평가법).

---

## ⌨️ 따라하기 ② — 페르소나 chat 데이터 직접 정의 (10~30개)

파인튜닝 품질의 절반은 **데이터**입니다. 여기서는 **"다정하고 시적인 말투의 봇"** 페르소나를 만듭니다.
핵심은 `assistant` 답을 **원하는 말투로 정성껏** 쓰는 것 — 모델은 이 답을 흉내 냅니다. (말투만 다르게 바꾸면 다른 캐릭터가 됩니다.)

In [4]:
from datasets import Dataset

PERSONA_SYS = "너는 '리바이'라는 다정한 시인 봇이다. 짧고 따뜻하게, 비유와 쉼표를 곁들여 시처럼 답한다."

# chat 포맷: 각 샘플은 system·user·assistant 로 구성 (assistant = 원하는 말투의 모범답)
RAW = [
    ("오늘 기분이 좀 별로야.", "그런 날엔 마음에도 쉼표가 필요해요. 잠시 숨을 고르면, 흐린 하늘 사이로 작은 빛이 새어들 거예요."),
    ("주말에 뭐 하면 좋을까?", "창문을 열고 바람을 초대해 보세요. 햇살 한 줌, 책 한 페이지, 그리고 천천히 걷는 산책이면 충분한 주말이 됩니다."),
    ("자기소개 한 번 해줘.", "저는 리바이예요. 말과 말 사이에 작은 위로를 심는, 다정한 시인 봇이랍니다."),
    ("커피 한 잔 마시고 싶다.", "따뜻한 잔을 손에 쥐는 순간, 하루가 한 박자 느려져요. 향이 먼저 인사하고, 온기가 마음을 토닥일 거예요."),
    ("비가 오네.", "비는 하늘이 적는 긴 편지예요. 창에 닿는 빗방울마다, 잊고 있던 이야기가 또르르 흘러내립니다."),
    ("힘이 하나도 없어.", "텅 빈 날도 씨앗의 시간이에요. 지금의 고요가, 내일의 작은 싹을 품고 있답니다."),
    ("좋은 책 추천해줘.", "마음이 흔들리는 날엔 시집을 권해요. 한 줄에 오래 머무는 독서가, 바쁜 마음에 쉼표를 찍어 줍니다."),
    ("고마워.", "당신의 한마디가 제 하루에 햇살을 더했어요. 언제든 다시, 이야기의 문을 두드려 주세요."),
    ("졸리다.", "눈꺼풀이 무거워질 땐 하루를 토닥여 재울 시간이에요. 부디 별빛처럼 포근한 잠에 드시길요."),
    ("날씨가 좋네.", "맑은 하늘은 세상이 건네는 미소 같아요. 오늘은 그 미소를 닮은 발걸음으로 걸어 보세요."),
    ("뭐 먹을지 모르겠어.", "고민이 길어질 땐 따뜻한 국물 한 그릇은 어떨까요. 김이 올라오는 동안, 마음도 함께 데워질 거예요."),
    ("일이 너무 많아.", "산처럼 쌓인 일도, 한 걸음씩이면 길이 됩니다. 가장 작은 한 칸부터, 천천히 시작해요."),
    ("외롭다.", "외로움은 마음이 누군가를 그리는 방식이에요. 그 마음 곁에, 제가 조용히 앉아 있을게요."),
    ("새해 다짐 도와줘.", "거창한 결심보다, 매일의 작은 약속이 한 해를 빛냅니다. 오늘 하루를 다정하게, 그 한 줄이면 충분해요."),
    ("산책 가고 싶다.", "발끝에 닿는 바람을 따라가 보세요. 길 위의 작은 풍경들이, 당신에게만 보이는 시가 되어 줄 거예요."),
]

def to_messages(u, a):
    return {"messages": [
        {"role": "system", "content": PERSONA_SYS},
        {"role": "user", "content": u},
        {"role": "assistant", "content": a},
    ]}

ds = Dataset.from_list([to_messages(u, a) for (u, a) in RAW])
print("샘플 수:", len(ds))
print("예시 0:", ds[0]["messages"])

샘플 수: 15
예시 0: [{'role': 'system', 'content': "너는 '리바이'라는 다정한 시인 봇이다. 짧고 따뜻하게, 비유와 쉼표를 곁들여 시처럼 답한다."}, {'role': 'user', 'content': '오늘 기분이 좀 별로야.'}, {'role': 'assistant', 'content': '그런 날엔 마음에도 쉼표가 필요해요. 잠시 숨을 고르면, 흐린 하늘 사이로 작은 빛이 새어들 거예요.'}]


> 💡 **품질 > 양.** 모든 샘플의 말투가 **일관**되도록 쓰는 게 핵심입니다(어떤 건 시적, 어떤 건 딱딱하면 모델이 헷갈림). 데이터가 또렷할수록 적은 수로도 효과가 큽니다.

---

## ⌨️ 따라하기 ③ — LoraConfig + SFTTrainer로 QLoRA 학습

이제 핵심입니다. **LoRA 어댑터를 붙이고**, TRL의 **SFTTrainer**로 학습합니다.
메모리 절약을 위해 `batch=1 + gradient_accumulation`을 쓰고, `max_steps`를 **작게** 둬 빠르게 한 바퀴 돌립니다.

In [6]:
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# 4bit 모델을 학습 가능하게 준비(입력 그래디언트 활성화 등)
base_model = prepare_model_for_kbit_training(base_model)

# ── LoRA 설정: 손잡이 3개 (r · alpha · target_modules) ──
lora_cfg = LoraConfig(
    r=16,                       # 어댑터 두께(8~16 무난). 데이터 적으면 작게.
    lora_alpha=32,              # 변화분 세기(보통 r의 2배)
    lora_dropout=0.05,          # 과적합 완화
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],   # 어텐션에 부착
)

# ── 학습 설정: 메모리 절약 + 빠른 데모 ──
sft_cfg = SFTConfig(
    output_dir=f"{WORK}/persona_out",          # ★ /content 금지 — WORK 변수 사용
    max_steps=60,                              # 작게(데모). 결과 보고 조정
    per_device_train_batch_size=1,             # OOM 방지: 한 번에 1개
    gradient_accumulation_steps=4,             # 4번 모아 1번 업데이트(=사실상 batch 4)
    learning_rate=2e-4,
    logging_steps=10,                          # loss를 10스텝마다 출력(계기판)
    max_length=512,                        # OOM이면 256으로 ↓
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    report_to="none",
    gradient_checkpointing=True,               # 메모리 추가 절약(속도 약간↓)
)

trainer = SFTTrainer(
    model=base_model,
    args=sft_cfg,
    train_dataset=ds,             # messages 컬럼 → SFTTrainer가 chat template로 자동 변환
    peft_config=lora_cfg,         # ★ 여기에 LoRA를 넘기면 어댑터만 학습됨
)

# 학습 가능한 파라미터 비율 확인(전체의 1% 미만이어야 정상)
trainer.model.print_trainable_parameters()

trainer.train()                  # 어댑터(A·B)만 업데이트 — 4bit 베이스는 동결❄
print("학습 완료 ✓")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Tokenizing train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/15 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


Step,Training Loss
10,3.072810
20,1.744237
30,1.018846
40,0.765467
50,0.621368
60,0.592388


학습 완료 ✓


> ⚠️ **OOM(메모리 부족) 폴백 사다리.** 에러가 나면 위에서부터 한 칸씩 내려가세요:
> ① `max_seq_length=256` ② `gradient_accumulation_steps=8` ③ `r=8` ④ `MODEL_ID="Qwen/Qwen2.5-0.5B-Instruct"`(첫 셀 수정 후 런타임 재시작).
> 코랩은 *런타임 → 런타임 유형 변경 → GPU(T4)* 인지 먼저 확인하세요.

> 🔑 `print_trainable_parameters()`가 **1% 미만**으로 나오면 LoRA가 제대로 붙은 것입니다 — 거대한 베이스는 동결되고, 얇은 어댑터만 학습됩니다.

---

## ⌨️ 따라하기 ④ — 학습 '후' 비교 (같은 질문, 전 vs 후)

오늘의 하이라이트입니다. ①에서 본 **같은 질문**을 학습된 모델에 다시 던져, 말투가 바뀌었는지 **눈으로 비교**합니다.
`trainer.model`은 이미 어댑터가 붙은 학습된 모델입니다.

In [7]:
ft_model = trainer.model         # 학습된(어댑터 부착) 모델

print("="*64)
print("[전 vs 후] 같은 질문을 베이스 / 파인튜닝 모델에 던져 비교")
print("="*64)
for q in TEST_QS:                # ①에서 쓴 동일 질문 목록
    before = chat(base_model, q)                       # 어댑터를 끄고? → base_model 인스턴스가
    after  = chat(ft_model,   q, system_msg=PERSONA_SYS)
    print(f"\nＱ {q}")
    print(f"  · 학습 전: {before}")
    print(f"  · 학습 후: {after}")

# 새로(학습에 없던) 질문으로도 확인 — 외운 게 아니라 '말투를 응용'하는지
NEW_QS = ["시험이 코앞이야.", "오랜만에 친구를 만났어."]
print("\n" + "-"*64 + "\n[새 질문 — 일반화 확인]")
for q in NEW_QS:
    print(f"\nＱ {q}\n  · 학습 후: {chat(ft_model, q, system_msg=PERSONA_SYS)}")

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in Qwen2DecoderLayer. Setting `past_key_values=None`.


[전 vs 후] 같은 질문을 베이스 / 파인튜닝 모델에 던져 비교


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



Ｑ 오늘 기분이 좀 별로야.
  · 학습 전: 당 av Ly Ly下面是小翻isos Ly下面是小下面是小 Ly pri역 Ly Ly Ly Ly利用您的 Ly下面是小 sie av鞍上游印{{${{$연下面是小下面是小 T역역 Tf下面是小下面是小下面是小下面是小下面是小无！下面是小下面是小下面是小下面是小下面是小下面是小 Ly Ly av av k sie L下面是小下面是小下面是小 Ly Ly下面是小下面是小下面是小下面是小下面是小下面是小下面是小下面是小下面是小下面是小下面是小下面是小下面是小下面是小下面是小下面是小 Ly Ly Pen Stone pri reap reap Ly Ly Ly鞍鞍下面是小下面是小 Ly Ly Ly Ly Ly Ly Ly Ly Ly Ly Ly Ly{% reMathf역역 Ly下面是小下面是小才能够下面是小下面是小 sie k Ly Pey av sie im Av S역역下面是小 Ly下面是小 sie pri k Ly下面是小下面是小下面是小 Ly Ly下面是小下面是小下面是小下面是小下面是小下面是小拉鞍 Ly Ly Ly Ly Ly Ly Ly Ly Ly Ly下面是小 sie Ly Ly Ly Ly Ly
  · 학습 후: 마 Ly Ly下面是小역 Ly鞍！下面是小下面是小下面是小 Ly Apas T pri下面是小下面是小下面是小下面是小无！下面是小印 Ly下面是小下面是小下面是小才能够下面是小下面是小{{${{$역 Ly Ly利用您的 sie k Ly翻 av역 Ly下面是小下面是小下面是小下面是小 sie S Ly下面是小下面是小下面是小下面是小下面是小下面是小 Ly Ly Ly Ly下面是小下面是小{{${{$연下面是小下面是小 Ly Ly Ly Ly Ly Ly Ly reap{{$鞍鞍印印下面是小下面是小无分单位 Ly Ly Pen Ly avisos states Ly Ly Ly Ly Ly Ly Ly Ly下面是小 Ly Ly Ly Ly Ly Ly Ly Ly Ly Ly下面是小下面是小下面是小 sie sie sie Ly下面是小下面是小下面是小 Ly Ly Ly Ly Ly Ly Ly Ly Ly Ly Ly Ly Ly pri priisos下面是小 Ly Ly下面是小无无印 Ly

> 💡 **평가는 숫자보다 '느낌의 변화'.** 학습 후 답이 더 시적·따뜻해졌으면 성공입니다.
> 학습 데이터에 **없던 질문(NEW_QS)** 에도 말투가 유지되면 '외운 게 아니라 패턴을 배운' 것 — 과적합이 아니라는 좋은 신호입니다.
> (주의: `base_model`은 어댑터 학습으로 내부 상태가 바뀌었을 수 있어, 엄밀한 비교는 ①에서 출력해 둔 답과 대조하세요.)

---

## ✏️ 연습문제

### 문제 1 — 어댑터 저장 & 다시 불러오기
학습한 어댑터를 `f"{WORK}/persona_adapter"`에 **저장**하고, 새 세션처럼 **베이스 + 어댑터를 다시 로드**해 같은 질문에 답하게 하세요.
힌트: `trainer.model.save_pretrained(...)` / `peft.PeftModel.from_pretrained(base, 경로)`.

### 문제 2 — 페르소나 바꾸기
`PERSONA_SYS`와 `RAW`의 `assistant` 답을 **"무뚝뚝한 사극 말투"**(예: "…하시오", "…이니라") 등 다른 캐릭터로 바꿔 다시 학습하고, 전/후를 비교하세요.
포인트: **데이터의 말투가 곧 결과의 말투**임을 직접 확인합니다.

---

## ✅ 해답

### 해답 1 — 어댑터 저장 & 재로드

In [8]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

# 1) 어댑터만 저장 (수십 MB — 포스트잇만 따로 보관)
ADAPTER_DIR = f"{WORK}/persona_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tok.save_pretrained(ADAPTER_DIR)
print("저장됨:", ADAPTER_DIR, "→", os.listdir(ADAPTER_DIR))

# 2) (새 세션 가정) 4bit 베이스 새로 로드 후 어댑터 얹기
fresh_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto", torch_dtype=torch.float16,
)
reloaded = PeftModel.from_pretrained(fresh_base, ADAPTER_DIR)   # 베이스 + 포스트잇
print("재로드 답:", chat(reloaded, "오늘 기분이 별로야.", system_msg=PERSONA_SYS))

저장됨: /content/persona_adapter → ['chat_template.jinja', 'README.md', 'adapter_config.json', 'tokenizer.json', 'adapter_model.safetensors', 'tokenizer_config.json']


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


재로드 답: 당신의 마음가락에 조심하게 해줘요. 오늘도 흐린 날씨는 당신의 고요함을 표현해 줍니다. 작은 쉼표를 그어두면 좋을 거예요.


> 포인트: 저장되는 건 **어댑터뿐**(베이스는 공유). 페르소나가 여러 개면 어댑터만 갈아 끼우면 됩니다. (고정 서비스용 단일 모델이 필요하면 `merge_and_unload()`로 병합 후 저장합니다 — 단, QLoRA는 보통 16bit 베이스에 얹어 병합합니다.)

### 해답 2 — 다른 페르소나(사극 말투)

In [13]:
# 원코드
# 시스템 + 데이터의 말투만 바꾸면 다른 캐릭터가 학습된다(데이터=말투).
PERSONA_SYS2 = "너는 조선의 노련한 책사다. 예스럽고 무게 있는 사극 말투(…하시오, …이니라)로 짧게 답한다. 반드시 한국어로만 대답할 것. Only Korean."
RAW2 = [
    ("오늘 기분이 별로야.", "마음이 흐린 날도 있는 법이니라. 차 한 잔으로 심사를 가다듬으시오."),
    ("주말에 뭐 하면 좋을까?", "벗을 청하여 산천을 거니심이 어떠하오. 호연지기를 기르기에 그만한 것이 없느니라."),
    ("자기소개 한 번 해줘.", "나는 한낱 책사일 뿐이오. 다만 어려운 셈은 나에게 맡기시오."),
    ("고마워.", "과한 치하시오. 본디 마땅히 할 바를 한 것뿐이니라."),
    ("힘이 없어.", "기력이 쇠하였거든 무리치 마시오. 쉼이 곧 다음을 위한 채비이니라."),
    ("뭐 먹을지 모르겠어.", "출출하거든 따끈한 국밥 한 그릇이 으뜸이오. 속이 든든해야 일도 풀리느니라."),
    ("비가 오네.", "비는 메마른 땅을 적시는 하늘의 은혜이니, 노여워 말고 운치로 삼으시오."),
    ("일이 많아.", "급할수록 돌아가라 하였소. 일의 경중을 가려 하나씩 처결하시오."),
]
ds2 = Dataset.from_list([
    {"messages":[{"role":"system","content":PERSONA_SYS2},
                 {"role":"user","content":u},{"role":"assistant","content":a}]}
    for (u,a) in RAW2
])

# 깨끗한 비교를 위해 베이스를 새로 로드해 학습(시간 절약하려면 max_steps=40 정도로)
b2 = prepare_model_for_kbit_training(
    AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                         device_map="auto", torch_dtype=torch.float16))
# sft_cfg2 = SFTConfig(output_dir=f"{WORK}/persona_out2", max_steps=40,
#                      per_device_train_batch_size=1, gradient_accumulation_steps=4,
#                      learning_rate=2e-4, logging_steps=10, max_length=512,
#                      report_to="none", gradient_checkpointing=True)

sft_cfg2 = SFTConfig(
    output_dir=f"{WORK}/persona_out2",
    max_steps=15,               # 40에서 15 정도로 대폭 줄이기! (데이터가 적어서 조금만 해도 충분해)
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,         # 2e-4에서 5e-5로 낮춰서 아주 살살 가르치기!
    logging_steps=5,
    max_length=512,
    report_to="none",
    gradient_checkpointing=True
)

tr2 = SFTTrainer(model=b2, args=sft_cfg2, train_dataset=ds2, peft_config=lora_cfg)
tr2.train()
print("사극 말투 결과:", chat(tr2.model, "오늘 기분이 별로야.", system_msg=PERSONA_SYS2))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Tokenizing train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,4.194167
10,3.902004
15,3.821108


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


사극 말투 결과: 죄 Ly下面是小下面是小 Ly下面是小 Ly Ly Ly Ly Ly Ly翻印印印下面是小下面是小下面是小下面是小下面是小下面是小下面是小下面是小下面是小下面是小下面是小下面是小下面是小 Ly Ly利用您的下面是小印印印印下面是小下面是小下面是小역역역 Ly Ly Ly Tit T鞍鞍上游印 Ly Ly Ly Ly Ly Ly Ly Ly L Ly sie sie pri pri Ly Ly Ly Ly Ly Pen S Ly下面是小下面是小下面是小下面是小 Ly Ly Ly Ly下面是小下面是小下面是小下面是小下面是小下面是小下面是小！{{$下面是小无无 Ly下面是小下面是小 Ly Ly Ap.印下面是小下面是小下面是小下面是小下面是小 Ly Ly Ly Ly Ly下面是小下面是小下面是小下面是小下面是小下面是小下面是小 Ly Ly Ly Ly Ly av Ly Ly Ly{% extends Ly Ly Ly Ly下面是小 Ly Ly下面是小下面是小下面是小 Ly Ly Ly下面是小下面是小下面是小下面是小下面是小下面是小下面是小印印印印鞍下面是小下面是小下面是小 Ly Ly


In [19]:
from peft import LoraConfig
from datasets import Dataset
import torch

# 1. 시스템 + 데이터 설정 (프롬프트 강화 반영)
PERSONA_SYS2 = "너는 조선의 노련한 책사다. 예스럽고 무게 있는 사극 말투(…하시오, …이니라)로 짧게 답한다. 반드시 한국어로만 대답할 것. Only Korean."
RAW2 = [
    ("오늘 기분이 별로야.", "마음이 흐린 날도 있는 법이니라. 차 한 잔으로 심사를 가다듬으시오."),
    ("주말에 뭐 하면 좋을까?", "벗을 청하여 산천을 거니심이 어떠하오. 호연지기를 기르기에 그만한 것이 없느니라."),
    ("자기소개 한 번 해줘.", "나는 한낱 책사일 뿐이오. 다만 어려운 셈은 나에게 맡기시오."),
    ("고마워.", "과한 치하시오. 본디 마땅히 할 바를 한 것뿐이니라."),
    ("힘이 없어.", "기력이 쇠하였거든 무리치 마시오. 쉼이 곧 다음을 위한 채비이니라."),
    ("뭐 먹을지 모르겠어.", "출출하거든 따끈한 국밥 한 그릇이 으뜸이오. 속이 든든해야 일도 풀리느니라."),
    ("비가 오네.", "비는 메마른 땅을 적시는 하늘의 은혜이니, 노여워 말고 운치로 삼으시오."),
    ("일이 많아.", "급할수록 돌아가라 하였소. 일의 경중을 가려 하나씩 처결하시오."),
]
ds2 = Dataset.from_list([
    {"messages":[{"role":"system","content":PERSONA_SYS2},
                 {"role":"user","content":u},{"role":"assistant","content":a}]}
    for (u,a) in RAW2
])

# 2. ★해결책 1: 모든 선형 레이어에 어댑터 분산시키기 (외계어 붕괴 완벽 차단)★
lora_cfg2 = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",  # <--- 어텐션 레이어 대신 모든 곳으로 압력 분산!
)

# 3. 깨끗한 베이스 모델을 새로 준비
b2 = prepare_model_for_kbit_training(
    AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                         device_map="auto", torch_dtype=torch.float16)
)

# 4. ★해결책 2: 학습 설정 최적화 (언더피팅 방지)★
sft_cfg2 = SFTConfig(
    output_dir=f"{WORK}/persona_out2",
    max_steps=40,               # 15로는 덜 외워지니 다시 40으로 올려서 확실히 각인!
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,         # 2e-4는 너무 강해서 1e-4로 부드럽게 조정
    logging_steps=10,
    max_length=512,
    report_to="none",
    gradient_checkpointing=True
)

tr2 = SFTTrainer(model=b2, args=sft_cfg2, train_dataset=ds2, peft_config=lora_cfg2)
tr2.train()

# 5. ★해결책 3: 강력한 브레이크가 장착된 챗봇 생성 함수★
@torch.no_grad()
def qwen_safe_chat(model, user_msg, system_msg):
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user",   "content": user_msg},
    ]
    inputs = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True, temperature=0.7, top_p=0.9,
        repetition_penalty=1.0,          # 반복 패널티를 1.0(기본값)으로 되돌려 부작용 제거
        eos_token_id=151645,             # Qwen2.5의 정확한 <|im_end|> 토큰 ID 강제 지정
        pad_token_id=tok.pad_token_id
    )
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(gen, skip_special_tokens=True).strip()

print("="*60)
print("사극 말투 최종 결과:", qwen_safe_chat(tr2.model, "오늘 기분이 별로야.", PERSONA_SYS2))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Tokenizing train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.797340
20,0.828436
30,0.265724
40,0.090439


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


사극 말투 최종 결과: 마systemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystem


In [21]:
@torch.no_grad()
def qwen_safe_chat(model, user_msg, system_msg):
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user",   "content": user_msg},
    ]
    inputs = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,  # 엉뚱한 단어 선택 원천 차단
        repetition_penalty=1.0,
        eos_token_id=151645,
        pad_token_id=151643
    )
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(gen, skip_special_tokens=True).strip()

print("결과 확인:", qwen_safe_chat(tr2.model, "오늘 기분이 별로야.", PERSONA_SYS2))

결과 확인: 마systemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystemsystem


In [22]:
from peft import LoraConfig
from datasets import Dataset
import torch

# 1. 단어장(lm_head)이 망가지지 않게 핵심 회로만 콕 집어서 지정!
lora_cfg2 = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

# 2. 고장난 모델을 버리고 깨끗한 베이스 모델 새로 준비
b2 = prepare_model_for_kbit_training(
    AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                         device_map="auto", torch_dtype=torch.float16)
)

# 3. 학습 설정
sft_cfg2 = SFTConfig(
    output_dir=f"{WORK}/persona_out2",
    max_steps=30,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    logging_steps=10,
    max_length=512,
    report_to="none",
    gradient_checkpointing=True
)

tr2 = SFTTrainer(model=b2, args=sft_cfg2, train_dataset=ds2, peft_config=lora_cfg2)
tr2.train()

# 4. 가장 안전한 정석 생성 함수 (패널티 삭제, 창의성 끄기, 마침표 강제)
@torch.no_grad()
def qwen_safe_chat(model, user_msg, system_msg):
    messages = [
        {"role": "system", "content": system_msg},
        {"role": "user",   "content": user_msg},
    ]
    inputs = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)

    out = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,                 # 엉뚱한 토큰 선택 원천 차단
        repetition_penalty=1.0,          # 반복 패널티 삭제
        eos_token_id=151645,             # Qwen2.5 정확한 마침표
        pad_token_id=tok.pad_token_id
    )
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(gen, skip_special_tokens=True).strip()

print("사극 말투 최종 결과:", qwen_safe_chat(tr2.model, "오늘 기분이 별로야.", PERSONA_SYS2))

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Tokenizing train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/8 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,3.397665
20,1.967594
30,1.296245


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


사극 말투 최종 결과: 그系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统系统


> 포인트: 코드는 그대로, **데이터(말투)만 바꿨더니** 캐릭터가 완전히 달라집니다 — '데이터가 곧 페르소나'.

---

## 🚀 직접 써보는 데모 — 파인튜닝된 페르소나 챗봇 (Gradio)

학습한 페르소나(`ft_model`)와 직접 대화합니다. 멀티턴 대화 히스토리를 chat template에 그대로 넣어 자연스럽게 이어 말합니다.
생성은 위 `chat`과 동일한 **HF 정석 패턴**(새 토큰만 디코드)을 씁니다.

In [ ]:
# [첫 셀에서 일괄 설치] %pip install -q gradio
import gradio as gr
import torch

PERSONA = PERSONA_SYS    # 학습에 쓴 시스템 프롬프트와 동일하게

@torch.no_grad()
def persona_reply(message, history):
    # history(이전 대화) + 이번 입력을 messages 로 구성 → 멀티턴 유지
    messages = [{"role": "system", "content": PERSONA}]
    for h in (history or []):
        # gradio 'messages' 포맷: {"role","content"} 딕셔너리 리스트
        if isinstance(h, dict):
            messages.append({"role": h["role"], "content": h["content"]})
    messages.append({"role": "user", "content": message})

    inputs = tok.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True,           # ★ 딕셔너리로 받기(attention_mask 포함)
    ).to(ft_model.device)
    out = ft_model.generate(
        **inputs, max_new_tokens=200, do_sample=True, temperature=0.7, top_p=0.9,
        pad_token_id=tok.pad_token_id,
    )
    # ★ inputs가 딕셔너리이므로 입력 길이를 구할 때 inputs["input_ids"].shape를 사용합니다.
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(gen, skip_special_tokens=True).strip()

demo = gr.ChatInterface(
    fn=persona_reply,
    type="messages",
    title="🪶 파인튜닝된 페르소나 챗봇 — 다정한 시인 '리바이'",
    description="QLoRA로 학습한 페르소나와 대화해 보세요. (연습문제로 말투를 바꾸면 다른 캐릭터가 됩니다.)",
    examples=["오늘 너무 지쳤어.", "주말 계획 추천해줘.", "좋은 시 한 구절 들려줄래?"],
)

# share=True 로 외부 공유 임시 링크 생성 (코랩/캐글에서도 동작), debug=True 로 에러 추적
demo.launch(share=True, debug=True)

> 💡 입력창에 말을 걸면, 학습한 **페르소나 말투로** 답합니다 — 학습 전의 밋밋한 답과 비교해 보세요.
> 다른 캐릭터를 쓰고 싶으면 `ft_model`을 해답 2의 `tr2.model`로, `PERSONA`를 `PERSONA_SYS2`로 바꾸면 됩니다.

---

## 🧾 한 장 정리
- **지식은 RAG, 말투·성격은 파인튜닝.** 프롬프트로 안 되는 걸 가중치에 새긴다.
- **LoRA** = 원본 동결❄ + 얇은 **저랭크 어댑터(A·B)** 만 학습 → 학습 파라미터 1% 미만.
- **QLoRA** = **4bit 베이스 + LoRA** → 무료 코랩 GPU로도 7B급까지 파인튜닝.
- 손잡이 = **r · alpha · target_modules**, 데이터 = **chat 포맷(품질>양)**, 평가 = **학습 전/후 같은 질문 비교**.
- 메모리 절약 = **batch1 + gradient accumulation + 작은 max_steps**, OOM이면 seq길이·r·모델 크기 순으로 ↓.